# 1. Segmentação de Instâncias com YOLOv8

## 1.1 Objetivo
Este notebook explora a **Segmentação de Instâncias**. Diferente da detecção de objetos simples (que apenas desenha um retângulo em volta do objeto), a segmentação identifica o contorno exato do objeto, pixel a pixel.

Você aprenderá a:
1. Realizar inferência de segmentação em imagens e vídeos.
2. Acessar as máscaras binárias (a forma precisa do objeto).
3. Calcular a área ocupada por um objeto.
4. Criar efeitos visuais avançados, como remover o fundo (Color Splash).

## 1.2 Detecção vs. Segmentação
- **Detecção (Object Detection)**: Retorna coordenadas (x1, y1, x2, y2) de uma caixa. É rápido, mas inclui o fundo dentro da caixa.
- **Segmentação (Instance Segmentation)**: Retorna uma "máscara" (polígono ou bitmap) que diz exatamente quais pixels pertencem ao objeto.

# 2. Primeiros Passos

Vamos carregar o modelo de segmentação (`yolo11n-seg.pt`) e testá-lo em uma imagem estática.

In [ ]:
from ultralytics import YOLO

# Carregar o modelo YOLO pré-treinado para segmentação
model = YOLO('yolo11n-seg.pt')

# Caminho da imagem (usaremos a imagem padrão do exemplo anterior ou uma nova)
source = 'original/image-do.jpg'

# Realizar inferência e salvar o resultado
results = model(source, save=True, conf=0.5)

# 3. Detecção em Tempo Real (Webcam)

O funcionamento básico na webcam é idêntico ao da detecção, mas o método `plot()` agora desenhará as máscaras coloridas sobre os objetos.

In [ ]:
import cv2
from ultralytics import YOLO

model = YOLO('yolo11n-seg.pt')
cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Erro ao acessar a webcam.")
    exit()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame)

    # Desenha as caixas E as máscaras
    annotated_frame = results[0].plot()

    cv2.imshow("YOLOv8 Segmentacao", annotated_frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# 4. Aplicação Avançada 1: Cálculo de Área

Como temos a máscara exata, podemos contar quantos pixels um objeto ocupa. Isso é útil para controle de qualidade industrial (ex: saber se um biscoito está do tamanho certo) ou monitoramento agrícola.

O código abaixo itera sobre as máscaras e imprime a área (em pixels) de cada objeto detectado.

In [ ]:
results = model(frame) # usando o último frame capturado

if results[0].masks is not None:
    # Itera sobre cada objeto detectado
    for i, mask in enumerate(results[0].masks.data):
        # mask é um tensor binário (0 = fundo, 1 = objeto)
        # Contamos o número de pixels não-zero
        area_pixels = mask.count_nonzero().item()
        
        # Pegamos a classe correspondente
        cls_id = int(results[0].boxes.cls[i])
        nome = model.names[cls_id]
        
        print(f"Objeto: {nome} | Área: {area_pixels} pixels")

# 5. Aplicação Avançada 2: Efeito Visual (Color Splash)

Vamos criar um efeito onde **todo o fundo fica em preto e branco**, e **apenas as pessoas detectadas permanecem coloridas**.

### Lógica:
1. Obter a máscara combinada de todas as pessoas.
2. Criar uma versão preto e branco (Grayscale) do frame original.
3. Onde a máscara for 1 (pessoa), usar o pixel colorido.
4. Onde a máscara for 0 (fundo), usar o pixel preto e branco.

In [ ]:
import cv2
import numpy as np
from ultralytics import YOLO

model = YOLO('yolo11n-seg.pt')
cap = cv2.VideoCapture(0)

while True:
    ret, frame = cap.read()
    if not ret: break

    # Reduzir resolução para processamento mais rápido (opcional)
    frame = cv2.resize(frame, (640, 480))
    
    # Inferência (apenas para a classe 'person', id=0)
    results = model(frame, classes=[0], verbose=False)

    # Se houver máscaras detectadas
    if results[0].masks is not None:
        # masks.data contém todas as máscaras. 
        # Somamos todas para ter uma máscara geral (alguma pessoa presente)
        # .max(0) comprime todas as detecções em uma única camada 2D
        masks_tensor = results[0].masks.data
        combined_mask = masks_tensor.max(0)[0].cpu().numpy()

        # A máscara do YOLO pode ser menor que a imagem original, redimensionamos para o tamanho do frame
        combined_mask = cv2.resize(combined_mask, (frame.shape[1], frame.shape[0]))

        # Processamento de Imagem
        # 1. Criar fundo Grayscale (convertido de volta para BGR para ter 3 canais)
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        gray_bgr = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)

        # 2. Binarizar a máscara (garantir 0 ou 1)
        # Pixels > 0.5 viram 1 (Objeto), outros 0 (Fundo)
        mask_binary = (combined_mask > 0.5).astype(np.uint8)
        
        # Expandir dimensões da máscara para 3 canais (para multiplicar com imagem colorida)
        mask_3ch = cv2.merge([mask_binary, mask_binary, mask_binary])

        # 3. Combinar: (Imagem Colorida * Máscara) + (Imagem PB * Máscara Invertida)
        # Parte colorida (onde mascara é 1)
        foreground = cv2.bitwise_and(frame, frame, mask=mask_binary)
        
        # Parte PB (onde mascara é 0)
        background = cv2.bitwise_and(gray_bgr, gray_bgr, mask=(1 - mask_binary))

        # Resultado final
        final_frame = cv2.add(foreground, background)
        
        cv2.imshow("Efeito Color Splash", final_frame)
    else:
        # Se não detectar ninguém, mostra apenas o frame original (ou PB, se preferir)
        cv2.imshow("Efeito Color Splash", frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# 6. Conclusão

## 6.1 Resumo
Neste notebook, vimos como a Segmentação de Instâncias nos dá um controle muito mais fino sobre os pixels da imagem. Diferente da caixa retangular, a máscara nos permite:
- Calcular áreas precisas.
- Recortar objetos e manipular o fundo.
- Criar efeitos de realidade aumentada simples.

## 6.2 Próximos Passos
- Tente inverter a lógica do Color Splash: Deixar o fundo colorido e a pessoa em preto e branco.
- Tente substituir o fundo (background replacement) por uma imagem virtual (como em chamadas de vídeo).